# 08. Hyperparameter Tuning Using Grid Search and Random Search

## 📚 Learning Objectives

By completing this notebook, you will:
- Perform hyperparameter tuning
- Use Grid Search
- Use Random Search
- Optimize model parameters
- Compare search strategies

## 🔗 Prerequisites

- ✅ Understanding of hyperparameters
- ✅ Understanding of model tuning
- ✅ Scikit-learn knowledge

---

This notebook covers practical activities from **Course 05, Unit 4**:
- Hyperparameter tuning using techniques like Grid Search and Random Search

---

## Introduction

**Hyperparameter tuning** optimizes model performance by systematically searching for the best parameter values using Grid Search or Random Search techniques.

## The Story

**BEFORE**: You can train models but don't know how to optimize their performance.

**AFTER**: You'll learn hyperparameter tuning: grid search, random search, and finding the best model parameters!

**Why this matters**: Hyperparameter Tuning Using Grid Search and Random Search is essential for building complete, professional data science solutions!

---

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Model, param grid
- sklearn

**Outputs:** What you'll see when you run the cells

- Best params
- CV results
- Printed summary

---

In [1]:
# Imports
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     cross_val_score, train_test_split)
from scipy.stats import randint

print("✅ Libraries imported!")
print("\nHyperparameter Tuning: Grid Search and Random Search")
print("=" * 60)
print("""
Hyperparameters are the knobs YOU set before training (n_estimators,
max_depth, ...), unlike parameters the model learns (coefficients, splits).

Remember Example 07's dead end: model selection cost =
  (parameter combinations) x (CV folds) x (per-fit time).
This notebook runs that search for real, twice:
  - Grid search: try EVERY combination
  - Random search: sample a fixed number of combinations
""")

✅ Libraries imported!

Hyperparameter Tuning: Grid Search and Random Search

Hyperparameters are the knobs YOU set before training (n_estimators,
max_depth, ...), unlike parameters the model learns (coefficients, splits).

Remember Example 07's dead end: model selection cost =
  (parameter combinations) x (CV folds) x (per-fit time).
This notebook runs that search for real, twice:
  - Grid search: try EVERY combination
  - Random search: sample a fixed number of combinations



## Part 1: Baseline — Default Hyperparameters

Before tuning anything, measure what the default model gives us.

In [2]:
print("Part 1: Baseline with default hyperparameters")
print("-" * 60)

X, y = make_classification(n_samples=2000, n_features=15, n_informative=6,
                           n_redundant=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Dataset: {X.shape[0]} samples x {X.shape[1]} features")

start = time.time()
baseline_scores = cross_val_score(
    RandomForestClassifier(random_state=42), X_train, y_train, cv=3)
baseline_time = time.time() - start
baseline_score = baseline_scores.mean()
print(f"\nDefault RandomForest, 3-fold CV accuracy: {baseline_score:.4f}")
print(f"Time for the baseline evaluation: {baseline_time:.2f}s")

Part 1: Baseline with default hyperparameters
------------------------------------------------------------
Dataset: 2000 samples x 15 features



Default RandomForest, 3-fold CV accuracy: 0.9167
Time for the baseline evaluation: 0.27s


## Part 2: Grid Search — Try Every Combination

`GridSearchCV` cross-validates **every** combination in the grid and refits
the best one on the full training set.

In [3]:
print("Part 2: Grid Search")
print("-" * 60)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, None],
    'min_samples_split': [2, 5],
}
n_combos = 2 * 2 * 2
n_folds = 3
print(f"Grid: {param_grid}")
print(f"Cost: {n_combos} combinations x {n_folds} folds (+1 refit) = {n_combos * n_folds + 1} fits")

start = time.time()
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=n_folds, n_jobs=-1)
grid.fit(X_train, y_train)
grid_time = time.time() - start

print(f"\nGrid search finished in {grid_time:.2f}s")
print(f"Best parameters: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}  (baseline was {baseline_score:.4f})")
grid_test = grid.score(X_test, y_test)
print(f"Held-out test accuracy of the tuned model: {grid_test:.4f}")

Part 2: Grid Search
------------------------------------------------------------
Grid: {'n_estimators': [50, 100], 'max_depth': [5, None], 'min_samples_split': [2, 5]}
Cost: 8 combinations x 3 folds (+1 refit) = 25 fits



Grid search finished in 1.95s
Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV accuracy: 0.9167  (baseline was 0.9167)
Held-out test accuracy of the tuned model: 0.9300


## Part 3: Random Search — Sample the Space

With many knobs, the grid explodes combinatorially. `RandomizedSearchCV`
samples a fixed budget of combinations from distributions instead.

In [4]:
print("Part 3: Random Search")
print("-" * 60)

param_dist = {
    'n_estimators': randint(30, 150),
    'max_depth': randint(3, 15),
    'min_samples_split': randint(2, 10),
}
n_iter = 8
print(f"Distributions: n_estimators ~ [30,150), max_depth ~ [3,15), min_samples_split ~ [2,10)")
print(f"Budget: {n_iter} sampled combinations x {n_folds} folds (+1 refit) = {n_iter * n_folds + 1} fits")

start = time.time()
rand = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                          param_dist, n_iter=n_iter, cv=n_folds,
                          random_state=42, n_jobs=-1)
rand.fit(X_train, y_train)
rand_time = time.time() - start

print(f"\nRandom search finished in {rand_time:.2f}s")
print(f"Best parameters: {rand.best_params_}")
print(f"Best CV accuracy: {rand.best_score_:.4f}")
rand_test = rand.score(X_test, y_test)
print(f"Held-out test accuracy of the tuned model: {rand_test:.4f}")

Part 3: Random Search
------------------------------------------------------------
Distributions: n_estimators ~ [30,150), max_depth ~ [3,15), min_samples_split ~ [2,10)
Budget: 8 sampled combinations x 3 folds (+1 refit) = 25 fits



Random search finished in 0.50s
Best parameters: {'max_depth': 10, 'min_samples_split': 6, 'n_estimators': 129}
Best CV accuracy: 0.9200
Held-out test accuracy of the tuned model: 0.9260


In [5]:
print("=" * 60)
print("Summary: Grid vs Random (measured on THIS machine)")
print("=" * 60)
print(f"""
Strategy         Fits   Time     Best CV acc   Test acc
Baseline (none)   {n_folds}    {baseline_time:5.2f}s   {baseline_score:.4f}        -
Grid search      {n_combos * n_folds + 1:3d}    {grid_time:5.2f}s   {grid.best_score_:.4f}        {grid_test:.4f}
Random search    {n_iter * n_folds + 1:3d}    {rand_time:5.2f}s   {rand.best_score_:.4f}        {rand_test:.4f}

Takeaways (from the numbers above, not from folklore):
  - Both searches explore the cost multiplication from Example 07 first-hand
  - Grid search is exhaustive but its cost grows multiplicatively per knob
  - Random search caps the budget (n_iter) regardless of how many knobs exist
  - Tuning gains on an easy dataset are modest - the WORKFLOW is the lesson

Next: Example 09 switches to unsupervised learning (no labels at all).
""")
print("✅ Hyperparameter tuning with grid and random search - complete!")

Summary: Grid vs Random (measured on THIS machine)

Strategy         Fits   Time     Best CV acc   Test acc
Baseline (none)   3     0.27s   0.9167        -
Grid search       25     1.95s   0.9167        0.9300
Random search     25     0.50s   0.9200        0.9260

Takeaways (from the numbers above, not from folklore):
  - Both searches explore the cost multiplication from Example 07 first-hand
  - Grid search is exhaustive but its cost grows multiplicatively per knob
  - Random search caps the budget (n_iter) regardless of how many knobs exist
  - Tuning gains on an easy dataset are modest - the WORKFLOW is the lesson

Next: Example 09 switches to unsupervised learning (no labels at all).

✅ Hyperparameter tuning with grid and random search - complete!
